# 02 DFSI Reproduction

This notebook attempts to reproduce the published District Flood Severity Index (DFSI) for **Robust Flood-Risk Prioritisation in India**.

The reproduction is deliberately conservative. It documents every formula component, records all joins, prefers LGD identifiers where they are available, and keeps ambiguous or unmatched records in audit tables instead of silently forcing a match.

## Reproduction Formula

The corrected published equation is implemented in log space:

```text
calculated_dfsi =
    log10(1 + flood_events)
  - log10(1 + mean_flood_duration)
  - 2 * log10(1 + fatalities)
  - log10(1 + injured)
  - log10(1 + flooded_area)
  - log10(1 + population)
```

This is equivalent to taking `log10` of the corresponding positive float64 product/ratio, but avoids constructing an integer product first. No values are standardised, imputed, corrected, rescaled, or otherwise changed to improve agreement with `DFSI.csv`. The only exception is that districts with no matched event in the exploded event inventory are assigned `0` flood events for the count component.

## Setup

Paths are defined with `pathlib`. The setup searches upward from the current working directory to find the repository root, so the notebook can be run from the repository root or from inside `notebooks`.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

EXPECTED_RAW_FILES = [
    "DFSI.csv",
    "District_FloodedArea.csv",
    "District_FloodImpact.csv",
    "India_Flood_Inventory_v3.csv",
]


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        raw_dir = candidate / "data" / "raw"
        if all((raw_dir / file_name).exists() for file_name in EXPECTED_RAW_FILES):
            return candidate
    raise FileNotFoundError(
        "Could not find data/raw with the expected CSV files. "
        "Start Jupyter from the repository root or a directory inside it."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / "data" / "raw"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    "dfsi": RAW_DIR / "DFSI.csv",
    "flooded_area": RAW_DIR / "District_FloodedArea.csv",
    "flood_impact": RAW_DIR / "District_FloodImpact.csv",
    "flood_inventory": RAW_DIR / "India_Flood_Inventory_v3.csv",
}

print(f"Project root: {PROJECT_ROOT}")
for name, path in PATHS.items():
    print(f"{name}: {path.relative_to(PROJECT_ROOT)}")

Project root: /Users/abhijithshaji/Documents/india-flood-index-robustness
dfsi: data/raw/DFSI.csv
flooded_area: data/raw/District_FloodedArea.csv
flood_impact: data/raw/District_FloodImpact.csv
flood_inventory: data/raw/India_Flood_Inventory_v3.csv


## Load Source Datasets

The raw files are read without modifying any values.

In [2]:
dfsi = pd.read_csv(PATHS["dfsi"], low_memory=False)
flooded_area = pd.read_csv(PATHS["flooded_area"], low_memory=False)
flood_impact = pd.read_csv(PATHS["flood_impact"], low_memory=False)
flood_inventory = pd.read_csv(PATHS["flood_inventory"], low_memory=False)

source_tables = {
    "DFSI.csv": dfsi,
    "District_FloodedArea.csv": flooded_area,
    "District_FloodImpact.csv": flood_impact,
    "India_Flood_Inventory_v3.csv": flood_inventory,
}

for name, table in source_tables.items():
    print(f"{name}: {table.shape[0]:,} rows x {table.shape[1]:,} columns")

DFSI.csv: 744 rows x 3 columns
District_FloodedArea.csv: 732 rows x 4 columns
District_FloodImpact.csv: 732 rows x 5 columns
India_Flood_Inventory_v3.csv: 6,876 rows x 23 columns


## Formula Component Mapping

The table below records which source columns are used for each DFSI component and why. This is an audit table, not a transformation recipe for cleaning the data.

In [3]:
component_mapping = pd.DataFrame([
    {
        "formula_component": "flood_events",
        "source_dataset": "India_Flood_Inventory_v3.csv",
        "source_column": "UEI, Districts, District_LGD_Codes",
        "selected_field": "count of distinct UEI after exploding district entries",
        "identifier_basis": "LGD district codes are parsed where present; name keys are required for joins to the district summary tables because those tables do not contain LGD codes",
        "notes": "Rows with missing districts/codes and name/code length mismatches are audited separately.",
    },
    {
        "formula_component": "mean_flood_duration",
        "source_dataset": "District_FloodImpact.csv",
        "source_column": "Mean_Flood_Duration",
        "selected_field": "Mean_Flood_Duration",
        "identifier_basis": "District name only; no state or LGD code is available in this source file",
        "notes": "Missing durations are not imputed.",
    },
    {
        "formula_component": "fatalities",
        "source_dataset": "District_FloodImpact.csv",
        "source_column": "Human_fatality",
        "selected_field": "Human_fatality",
        "identifier_basis": "District name only; no state or LGD code is available in this source file",
        "notes": "Used as provided, including zeros.",
    },
    {
        "formula_component": "injured",
        "source_dataset": "District_FloodImpact.csv",
        "source_column": "Human_injured",
        "selected_field": "Human_injured",
        "identifier_basis": "District name only; no state or LGD code is available in this source file",
        "notes": "Used as provided, including zeros.",
    },
    {
        "formula_component": "flooded_area",
        "source_dataset": "District_FloodedArea.csv",
        "source_column": "Corrected_Percent_Flooded_Area",
        "selected_field": "Corrected_Percent_Flooded_Area",
        "identifier_basis": "District name only; no state or LGD code is available in this source file",
        "notes": "The corrected field is selected because it appears to subtract permanent water from flooded area; `Percent_Flooded_Area` is retained in the comparison table for review.",
    },
    {
        "formula_component": "population",
        "source_dataset": "District_FloodImpact.csv",
        "source_column": "Population",
        "selected_field": "Population",
        "identifier_basis": "District name only; no state or LGD code is available in this source file",
        "notes": "Used on the raw scale supplied by the source file.",
    },
    {
        "formula_component": "published_dfsi",
        "source_dataset": "DFSI.csv",
        "source_column": "DFSI",
        "selected_field": "DFSI",
        "identifier_basis": "District name in `Unnamed: 0` plus `State_Name`; no LGD code is available",
        "notes": "The district column appears to be an unnamed saved index; spelling is preserved.",
    },
])

display(component_mapping)
component_mapping.to_csv(TABLE_DIR / "dfsi_component_mapping.csv", index=False)

,formula_component,source_dataset,source_column,selected_field,identifier_basis,notes
0,flood_events,India_Flood_Inventory_v3.csv,"UEI, Districts, District_LGD_Codes",count of distinct UEI after exploding district entries,LGD district codes are parsed where present; name keys are required for joins to the district summary tables because those tables do not contain LGD codes,Rows with missing districts/codes and name/code length mismatches are audited separately.
1,mean_flood_duration,District_FloodImpact.csv,Mean_Flood_Duration,Mean_Flood_Duration,District name only; no state or LGD code is available in this source file,Missing durations are not imputed.
2,fatalities,District_FloodImpact.csv,Human_fatality,Human_fatality,District name only; no state or LGD code is available in this source file,"Used as provided, including zeros."
3,injured,District_FloodImpact.csv,Human_injured,Human_injured,District name only; no state or LGD code is available in this source file,"Used as provided, including zeros."
4,flooded_area,District_FloodedArea.csv,Corrected_Percent_Flooded_Area,Corrected_Percent_Flooded_Area,District name only; no state or LGD code is available in this source file,The corrected field is selected because it appears to subtract permanent water from flooded area; `Percent_Flooded_Area` is retained in the comparison table...
5,population,District_FloodImpact.csv,Population,Population,District name only; no state or LGD code is available in this source file,Used on the raw scale supplied by the source file.
6,published_dfsi,DFSI.csv,DFSI,DFSI,District name in `Unnamed: 0` plus `State_Name`; no LGD code is available,The district column appears to be an unnamed saved index; spelling is preserved.


## Helper Functions

These helpers create review keys and parse comma-separated list columns. Review keys are used only to make joins auditable; they do not overwrite raw values.

In [4]:
def make_review_key(value):
    if pd.isna(value):
        return pd.NA
    return re.sub(r"\s+", " ", str(value).strip()).casefold()


def split_list_cell(value):
    if pd.isna(value):
        return []
    return [part.strip() for part in str(value).split(",")]


def clean_lgd_code(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    if text == "" or text.casefold() in {"none", "nan", "na", "<na>"}:
        return pd.NA
    return text

## Parse Flood Events by District

The event inventory is the only source with LGD district and state code fields. District-level event counts are therefore parsed with LGD codes where available. However, the district summary files and `DFSI.csv` do not include LGD codes, so downstream comparison still requires auditable district-name joins.

In [5]:
event_rows = []
event_parse_audit_records = []

for source_row_id, row in flood_inventory.reset_index(names="source_row_id").iterrows():
    districts = split_list_cell(row["Districts"])
    district_codes = split_list_cell(row["District_LGD_Codes"])
    if not districts:
        event_parse_audit_records.append({
            "source_row_id": row["source_row_id"],
            "UEI": row.get("UEI"),
            "issue_type": "missing_district_list",
            "district_count": 0,
            "district_lgd_code_count": len(district_codes),
            "State": row.get("State"),
            "State_Codes": row.get("State_Codes"),
        })
        continue

    if district_codes and len(districts) != len(district_codes):
        event_parse_audit_records.append({
            "source_row_id": row["source_row_id"],
            "UEI": row.get("UEI"),
            "issue_type": "district_name_lgd_code_count_mismatch",
            "district_count": len(districts),
            "district_lgd_code_count": len(district_codes),
            "State": row.get("State"),
            "State_Codes": row.get("State_Codes"),
        })

    for position, district_name in enumerate(districts):
        lgd_code = district_codes[position] if position < len(district_codes) else pd.NA
        event_rows.append({
            "source_row_id": row["source_row_id"],
            "UEI": row.get("UEI"),
            "district_position": position + 1,
            "district_name_from_inventory": district_name,
            "district_review_key": make_review_key(district_name),
            "district_lgd_code": clean_lgd_code(lgd_code),
            "event_state_text": row.get("State"),
            "event_state_codes_text": row.get("State_Codes"),
        })

event_districts = pd.DataFrame(event_rows)
event_parse_audit = pd.DataFrame(event_parse_audit_records)

print(f"Exploded event-district records: {len(event_districts):,}")
print(f"Event parse audit records: {len(event_parse_audit):,}")
display(event_districts.head())
display(event_parse_audit.head())

Exploded event-district records: 24,628
Event parse audit records: 59


,source_row_id,UEI,district_position,district_name_from_inventory,district_review_key,district_lgd_code,event_state_text,event_state_codes_text
0,3,UEI-IMD-FL-1967-0004,1,Bhadrak,bhadrak,348,"Odisha, Bihar, Uttar Pradesh, Madhya Pradesh, Rajasthan, Haryana","21,10,9,23,8,6"
1,3,UEI-IMD-FL-1967-0004,2,Dhenkanal,dhenkanal,352,"Odisha, Bihar, Uttar Pradesh, Madhya Pradesh, Rajasthan, Haryana","21,10,9,23,8,6"
2,3,UEI-IMD-FL-1967-0004,3,Jajapur,jajapur,356,"Odisha, Bihar, Uttar Pradesh, Madhya Pradesh, Rajasthan, Haryana","21,10,9,23,8,6"
3,3,UEI-IMD-FL-1967-0004,4,Subarnapur,subarnapur,<NA>,"Odisha, Bihar, Uttar Pradesh, Madhya Pradesh, Rajasthan, Haryana","21,10,9,23,8,6"
4,3,UEI-IMD-FL-1967-0004,5,Nuapada,nuapada,368,"Odisha, Bihar, Uttar Pradesh, Madhya Pradesh, Rajasthan, Haryana","21,10,9,23,8,6"


,source_row_id,UEI,issue_type,district_count,district_lgd_code_count,State,State_Codes
0,0,UEI-IMD-FL-1967-0001,missing_district_list,0,0,Assam,18
1,1,UEI-IMD-FL-1967-0002,missing_district_list,0,0,"Maharashtra, Gujarat","27,24"
2,2,UEI-IMD-FL-1967-0003,missing_district_list,0,0,Uttar Pradesh,9
3,4,UEI-IMD-FL-1968-0001,missing_district_list,0,0,Assam,18
4,5,UEI-IMD-FL-1968-0002,missing_district_list,0,0,West Bengal,19


## Event Count Aggregation

Event counts are calculated as distinct `UEI` values per district. LGD-code event counts are retained, but the reproduction comparison uses district-name event counts because the other district-level source files do not provide LGD identifiers.

In [6]:
event_counts_by_lgd = (
    event_districts.dropna(subset=["district_lgd_code"])
    .groupby("district_lgd_code", dropna=False)
    .agg(
        flood_events_lgd=("UEI", "nunique"),
        inventory_district_names=("district_name_from_inventory", lambda s: " | ".join(sorted(set(map(str, s.dropna()))))),
    )
    .reset_index()
)

event_counts_by_name = (
    event_districts.groupby("district_review_key", dropna=False)
    .agg(
        flood_events=("UEI", "nunique"),
        inventory_district_names=("district_name_from_inventory", lambda s: " | ".join(sorted(set(map(str, s.dropna()))))),
        lgd_codes_seen=("district_lgd_code", lambda s: " | ".join(sorted(set(map(str, s.dropna()))))),
        lgd_code_count=("district_lgd_code", lambda s: int(s.dropna().nunique())),
    )
    .reset_index()
)

name_lgd_ambiguities = event_counts_by_name.query("lgd_code_count > 1").copy()

display(event_counts_by_name.head())
display(name_lgd_ambiguities.head())

,district_review_key,flood_events,inventory_district_names,lgd_codes_seen,lgd_code_count
0,,51,,603,1
1,& parts of himachal pradesh,2,& Parts of Himachal Pradesh,,0
2,& parts of uttar pradesh,1,& Parts of Uttar Pradesh,,0
3,& various parts of nagaland,1,& various parts of Nagaland,,0
4,11 districts,1,11 districts,,0


,district_review_key,flood_events,inventory_district_names,lgd_codes_seen,lgd_code_count


## Prepare District Summary Sources

The flood-impact and flooded-area sources do not provide state names or LGD codes. Repeated district names are therefore potential many-to-many joins and are kept out of the primary one-to-one reproduction table.

In [7]:
impact = flood_impact.reset_index(names="impact_source_row_id").copy()
impact["district_review_key"] = impact["Dist_Name"].map(make_review_key)

area = flooded_area.reset_index(names="area_source_row_id").copy()
area["district_review_key"] = area["Dist_Name"].map(make_review_key)

published = dfsi.reset_index(names="dfsi_source_row_id").rename(columns={"Unnamed: 0": "DFSI_Dist_Name"}).copy()
published["district_review_key"] = published["DFSI_Dist_Name"].map(make_review_key)
published["state_review_key"] = published["State_Name"].map(make_review_key)

impact_name_counts = impact["district_review_key"].value_counts(dropna=False).rename("impact_name_count")
area_name_counts = area["district_review_key"].value_counts(dropna=False).rename("area_name_count")
published_name_counts = published["district_review_key"].value_counts(dropna=False).rename("published_name_count")

display(impact_name_counts[impact_name_counts > 1].reset_index().rename(columns={"index": "district_review_key"}))
display(area_name_counts[area_name_counts > 1].reset_index().rename(columns={"index": "district_review_key"}))
display(published_name_counts[published_name_counts > 1].reset_index().rename(columns={"index": "district_review_key"}))

,district_review_key,impact_name_count
0,balrampur,2
1,aurangabad,2
2,raigarh,2
3,bilaspur,2
4,pratapgarh,2
5,hamirpur,2


,district_review_key,area_name_count
0,balrampur,2
1,aurangabad,2
2,raigarh,2
3,bilaspur,2
4,pratapgarh,2
5,hamirpur,2


,district_review_key,published_name_count
0,balrampur,4
1,aurangabad,4
2,pratapgarh,4
3,bilaspur,4
4,hamirpur,4
5,raigarh,4


## Join Documentation

All joins are documented before calculating DFSI. Since most source files lack LGD identifiers, name-only joins are used only after identifying ambiguous keys and excluding multiply matched records from the primary reproduction table.

In [8]:
join_documentation = pd.DataFrame([
    {
        "join_step": 1,
        "left_dataset": "District_FloodImpact.csv",
        "right_dataset": "District_FloodedArea.csv",
        "join_keys": "district_review_key derived from Dist_Name",
        "join_type": "inner for calculable one-to-one records; ambiguous and unmatched records audited separately",
        "lgd_preference_status": "Cannot use LGD: neither source has LGD district/state codes",
        "ambiguity_policy": "Keys appearing more than once on either side are excluded from the primary calculation and saved to dfsi_multiply_matched_records.csv",
    },
    {
        "join_step": 2,
        "left_dataset": "Impact + FloodedArea one-to-one table",
        "right_dataset": "India_Flood_Inventory_v3.csv exploded event counts",
        "join_keys": "district_review_key derived from district names",
        "join_type": "left join; unmatched event counts audited separately",
        "lgd_preference_status": "Event counts are parsed by LGD where present, but summary sources lack LGD, so final join cannot be LGD-based",
        "ambiguity_policy": "District names mapping to multiple LGD codes are reported as possible identifier ambiguity",
    },
    {
        "join_step": 3,
        "left_dataset": "Calculated DFSI candidate table",
        "right_dataset": "DFSI.csv",
        "join_keys": "district_review_key derived from District_FloodImpact.Dist_Name and DFSI.csv unnamed district column",
        "join_type": "inner for comparison; unmatched and multiply matched records audited separately",
        "lgd_preference_status": "Cannot use LGD: DFSI.csv has district/state names but no LGD codes",
        "ambiguity_policy": "Repeated published district names are excluded from the primary comparison unless the join is one-to-one",
    },
])

display(join_documentation)
join_documentation.to_csv(TABLE_DIR / "dfsi_join_documentation.csv", index=False)

,join_step,left_dataset,right_dataset,join_keys,join_type,lgd_preference_status,ambiguity_policy
0,1,District_FloodImpact.csv,District_FloodedArea.csv,district_review_key derived from Dist_Name,inner for calculable one-to-one records; ambiguous and unmatched records audited separately,Cannot use LGD: neither source has LGD district/state codes,Keys appearing more than once on either side are excluded from the primary calculation and saved to dfsi_multiply_matched_records.csv
1,2,Impact + FloodedArea one-to-one table,India_Flood_Inventory_v3.csv exploded event counts,district_review_key derived from district names,left join; unmatched event counts audited separately,"Event counts are parsed by LGD where present, but summary sources lack LGD, so final join cannot be LGD-based",District names mapping to multiple LGD codes are reported as possible identifier ambiguity
2,3,Calculated DFSI candidate table,DFSI.csv,district_review_key derived from District_FloodImpact.Dist_Name and DFSI.csv unnamed district column,inner for comparison; unmatched and multiply matched records audited separately,Cannot use LGD: DFSI.csv has district/state names but no LGD codes,Repeated published district names are excluded from the primary comparison unless the join is one-to-one


## Build Join Audit Tables

Unmatched and multiply matched records are kept in separate tables for review.

In [9]:
all_keys = pd.Index([])
for key_series in [impact["district_review_key"], area["district_review_key"], published["district_review_key"], event_counts_by_name["district_review_key"]]:
    all_keys = all_keys.union(pd.Index(key_series.dropna().unique()))

key_audit = pd.DataFrame({"district_review_key": all_keys})
key_audit = key_audit.merge(impact_name_counts.reset_index().rename(columns={"index": "district_review_key"}), on="district_review_key", how="left")
key_audit = key_audit.merge(area_name_counts.reset_index().rename(columns={"index": "district_review_key"}), on="district_review_key", how="left")
key_audit = key_audit.merge(published_name_counts.reset_index().rename(columns={"index": "district_review_key"}), on="district_review_key", how="left")
key_audit = key_audit.merge(event_counts_by_name[["district_review_key", "flood_events", "lgd_code_count", "lgd_codes_seen"]], on="district_review_key", how="left")

for column in ["impact_name_count", "area_name_count", "published_name_count", "flood_events", "lgd_code_count"]:
    key_audit[column] = key_audit[column].fillna(0).astype(int)

unmatched_records = key_audit.query(
    "impact_name_count == 0 or area_name_count == 0 or published_name_count == 0 or flood_events == 0"
).copy()
unmatched_records["audit_note"] = unmatched_records.apply(
    lambda row: "; ".join([
        label for label, missing in [
            ("missing from District_FloodImpact", row["impact_name_count"] == 0),
            ("missing from District_FloodedArea", row["area_name_count"] == 0),
            ("missing from DFSI.csv", row["published_name_count"] == 0),
            ("no matched flood events in inventory", row["flood_events"] == 0),
        ] if missing
    ]),
    axis=1,
)

multiply_matched_records = key_audit.query(
    "impact_name_count > 1 or area_name_count > 1 or published_name_count > 1 or lgd_code_count > 1"
).copy()
multiply_matched_records["audit_note"] = multiply_matched_records.apply(
    lambda row: "; ".join([
        label for label, issue in [
            ("multiple District_FloodImpact rows", row["impact_name_count"] > 1),
            ("multiple District_FloodedArea rows", row["area_name_count"] > 1),
            ("multiple DFSI.csv rows", row["published_name_count"] > 1),
            ("inventory name maps to multiple LGD district codes", row["lgd_code_count"] > 1),
        ] if issue
    ]),
    axis=1,
)

display(unmatched_records.head())
display(multiply_matched_records.head())

unmatched_records.to_csv(TABLE_DIR / "dfsi_unmatched_records.csv", index=False)
multiply_matched_records.to_csv(TABLE_DIR / "dfsi_multiply_matched_records.csv", index=False)
event_parse_audit.to_csv(TABLE_DIR / "dfsi_event_parse_audit.csv", index=False)
name_lgd_ambiguities.to_csv(TABLE_DIR / "dfsi_inventory_name_lgd_ambiguities.csv", index=False)

,district_review_key,impact_name_count,area_name_count,published_name_count,flood_events,lgd_code_count,lgd_codes_seen,audit_note
0,,0,0,0,51,1,603,missing from District_FloodImpact; missing from District_FloodedArea; missing from DFSI.csv
1,& parts of himachal pradesh,0,0,0,2,0,,missing from District_FloodImpact; missing from District_FloodedArea; missing from DFSI.csv
2,& parts of uttar pradesh,0,0,0,1,0,,missing from District_FloodImpact; missing from District_FloodedArea; missing from DFSI.csv
3,& various parts of nagaland,0,0,0,1,0,,missing from District_FloodImpact; missing from District_FloodedArea; missing from DFSI.csv
4,11 districts,0,0,0,1,0,,missing from District_FloodImpact; missing from District_FloodedArea; missing from DFSI.csv


,district_review_key,impact_name_count,area_name_count,published_name_count,flood_events,lgd_code_count,lgd_codes_seen,audit_note
54,aurangabad,2,2,4,89,1,189,multiple District_FloodImpact rows; multiple District_FloodedArea rows; multiple DFSI.csv rows
77,balrampur,2,2,4,37,1,649,multiple District_FloodImpact rows; multiple District_FloodedArea rows; multiple DFSI.csv rows
133,bilaspur,2,2,4,62,1,375,multiple District_FloodImpact rows; multiple District_FloodedArea rows; multiple DFSI.csv rows
322,hamirpur,2,2,4,116,1,17,multiple District_FloodImpact rows; multiple District_FloodedArea rows; multiple DFSI.csv rows
690,pratapgarh,2,2,4,57,1,629,multiple District_FloodImpact rows; multiple District_FloodedArea rows; multiple DFSI.csv rows


## Construct One-to-One Reproduction Table

The primary comparison excludes district keys with multiple rows in any required district-level source. This avoids joining solely on district name when the name is ambiguous.

In [10]:
one_to_one_keys = key_audit.query(
    "impact_name_count == 1 and area_name_count == 1 and published_name_count == 1"
)["district_review_key"]

impact_one = impact[impact["district_review_key"].isin(one_to_one_keys)].copy()
area_one = area[area["district_review_key"].isin(one_to_one_keys)].copy()
published_one = published[published["district_review_key"].isin(one_to_one_keys)].copy()

candidate = (
    impact_one.merge(
        area_one,
        on="district_review_key",
        how="inner",
        suffixes=("_impact", "_area"),
        validate="one_to_one",
    )
    .merge(
        event_counts_by_name[["district_review_key", "flood_events", "inventory_district_names", "lgd_codes_seen", "lgd_code_count"]],
        on="district_review_key",
        how="left",
        validate="one_to_one",
    )
    .merge(
        published_one[["dfsi_source_row_id", "DFSI_Dist_Name", "State_Name", "state_review_key", "DFSI", "district_review_key"]],
        on="district_review_key",
        how="inner",
        validate="one_to_one",
    )
)

candidate["flood_events_missing_from_inventory"] = candidate["flood_events"].isna()
candidate["flood_events"] = candidate["flood_events"].fillna(0).astype(int)

print(f"One-to-one candidate records: {len(candidate):,}")
display(candidate.head())

One-to-one candidate records: 720


,impact_source_row_id,Dist_Name_impact,Human_fatality,Human_injured,Population,Mean_Flood_Duration,district_review_key,area_source_row_id,Dist_Name_area,Percent_Flooded_Area,Parmanent_Water,Corrected_Percent_Flooded_Area,flood_events,inventory_district_names,lgd_codes_seen,lgd_code_count,dfsi_source_row_id,DFSI_Dist_Name,State_Name,state_review_key,DFSI,flood_events_missing_from_inventory
0,0,Dadra & Nagar Haveli,10,0,618459,1.0,dadra & nagar haveli,0,Dadra & Nagar Haveli,3.614705,0.645537,2.969168,0,NaN,NaN,NaN,653,Dadra & Nagar Haveli,DADRA & NAGAR HAVE,dadra & nagar have,9.820211,True
1,1,Daman,0,0,190125,NaN,daman,1,Daman,2.853534,1.308996,1.544538,0,NaN,NaN,NaN,738,Daman,DAMAN & DIU,daman & diu,5.233751,True
2,2,Diu,0,0,67412,NaN,diu,2,Diu,5.799818,4.979982,0.819836,0,NaN,NaN,NaN,741,Diu,DAMAN & DIU,daman & diu,4.269142,True
3,3,North & Middle Andaman,0,0,101677,1.0,north & middle andaman,3,North & Middle Andaman,1.314817,0.000000,1.314817,0,NaN,NaN,NaN,726,North & Middle Andaman,ANDAMAN & NICOBAR,andaman & nicobar,7.462575,True
4,4,South Andaman,1,0,257900,2.0,south andaman,4,South Andaman,1.443452,0.000000,1.443452,0,NaN,NaN,NaN,703,South Andaman,ANDAMAN & NICOBAR,andaman & nicobar,8.360463,True


## Calculate DFSI in Log Space

All six formula components are explicitly converted with `pd.to_numeric(..., errors="coerce").astype("float64")`. Rows with missing non-event components remain missing. The float64 product is calculated only as a verification check against the log-space calculation.

In [11]:
comparison = candidate.copy()

# No matched inventory events means the district has zero parsed events for this count component.
comparison["flood_events"] = comparison["flood_events"].fillna(0)

comparison["component_flood_events"] = pd.to_numeric(comparison["flood_events"], errors="coerce").astype("float64")
comparison["component_mean_flood_duration"] = pd.to_numeric(comparison["Mean_Flood_Duration"], errors="coerce").astype("float64")
comparison["component_fatalities"] = pd.to_numeric(comparison["Human_fatality"], errors="coerce").astype("float64")
comparison["component_injured"] = pd.to_numeric(comparison["Human_injured"], errors="coerce").astype("float64")
comparison["component_flooded_area"] = pd.to_numeric(comparison["Corrected_Percent_Flooded_Area"], errors="coerce").astype("float64")
comparison["component_population"] = pd.to_numeric(comparison["Population"], errors="coerce").astype("float64")

component_columns = [
    "component_flood_events",
    "component_mean_flood_duration",
    "component_fatalities",
    "component_injured",
    "component_flooded_area",
    "component_population",
]

for column in component_columns:
    assert comparison[column].dropna().ge(0).all(), f"Negative values found in {column}"

complete_component_mask = comparison[component_columns].notna().all(axis=1)
comparison["dfsi_float64_product"] = (
    (1.0 + comparison["component_flood_events"])
    / (
        (1.0 + comparison["component_mean_flood_duration"])
        * (1.0 + comparison["component_fatalities"]) ** 2.0
        * (1.0 + comparison["component_injured"])
        * (1.0 + comparison["component_flooded_area"])
        * (1.0 + comparison["component_population"])
    )
)
assert comparison.loc[complete_component_mask, "dfsi_float64_product"].gt(0).all(), "Complete-row product must be positive"

comparison["dfsi_calculated"] = (
    np.log10(1.0 + comparison["component_flood_events"])
    - np.log10(1.0 + comparison["component_mean_flood_duration"])
    - 2.0 * np.log10(1.0 + comparison["component_fatalities"])
    - np.log10(1.0 + comparison["component_injured"])
    - np.log10(1.0 + comparison["component_flooded_area"])
    - np.log10(1.0 + comparison["component_population"])
)
comparison["dfsi_calculated_from_product"] = np.log10(comparison["dfsi_float64_product"])
comparison["logspace_product_abs_diff"] = (comparison["dfsi_calculated"] - comparison["dfsi_calculated_from_product"]).abs()
comparison["absolute_difference"] = (comparison["dfsi_calculated"] - comparison["DFSI"]).abs()
comparison["rank_difference"] = comparison["dfsi_calculated"].rank(ascending=False) - comparison["DFSI"].rank(ascending=False)

comparison_columns = [
    "district_review_key",
    "Dist_Name_impact",
    "DFSI_Dist_Name",
    "State_Name",
    "component_flood_events",
    "component_mean_flood_duration",
    "component_fatalities",
    "component_injured",
    "component_flooded_area",
    "Percent_Flooded_Area",
    "Parmanent_Water",
    "component_population",
    "dfsi_float64_product",
    "dfsi_calculated",
    "dfsi_calculated_from_product",
    "logspace_product_abs_diff",
    "DFSI",
    "absolute_difference",
    "rank_difference",
    "flood_events_missing_from_inventory",
    "lgd_codes_seen",
    "lgd_code_count",
]

comparison_table = comparison[comparison_columns].rename(columns={"DFSI": "dfsi_published"})
display(comparison_table.head())
comparison_table.to_csv(TABLE_DIR / "dfsi_reproduction_comparison.csv", index=False)

,district_review_key,Dist_Name_impact,DFSI_Dist_Name,State_Name,component_flood_events,component_mean_flood_duration,component_fatalities,component_injured,component_flooded_area,Percent_Flooded_Area,Parmanent_Water,component_population,dfsi_float64_product,dfsi_calculated,dfsi_calculated_from_product,logspace_product_abs_diff,dfsi_published,absolute_difference,rank_difference,flood_events_missing_from_inventory,lgd_codes_seen,lgd_code_count
0,dadra & nagar haveli,Dadra & Nagar Haveli,Dadra & Nagar Haveli,DADRA & NAGAR HAVE,0.0,1.0,10.0,0.0,2.969168,3.614705,0.645537,618459.0,1.683347e-09,-8.773826,-8.773826,0.000000e+00,9.820211,18.594038,-442.0,True,NaN,NaN
1,daman,Daman,Daman,DAMAN & DIU,0.0,NaN,0.0,0.0,1.544538,2.853534,1.308996,190125.0,NaN,NaN,NaN,NaN,5.233751,NaN,NaN,True,NaN,NaN
2,diu,Diu,Diu,DAMAN & DIU,0.0,NaN,0.0,0.0,0.819836,5.799818,4.979982,67412.0,NaN,NaN,NaN,NaN,4.269142,NaN,NaN,True,NaN,NaN
3,north & middle andaman,North & Middle Andaman,North & Middle Andaman,ANDAMAN & NICOBAR,0.0,1.0,0.0,0.0,1.314817,1.314817,0.000000,101677.0,2.124351e-06,-5.672774,-5.672774,0.000000e+00,7.462575,13.135348,-693.0,True,NaN,NaN
4,south andaman,South Andaman,South Andaman,ANDAMAN & NICOBAR,0.0,2.0,1.0,0.0,1.443452,1.443452,0.000000,257900.0,1.322397e-07,-6.878638,-6.878638,8.881784e-16,8.360463,15.239101,-632.0,True,NaN,NaN


## Reproduction Metrics

Metrics are calculated only for rows where both the published DFSI and the calculated DFSI are finite. If the literal formula produces no finite DFSI values, the metrics are reported as missing rather than forced.

In [12]:
valid_metric_rows = comparison_table.replace([np.inf, -np.inf], np.nan).dropna(subset=["dfsi_calculated", "dfsi_published"])

if len(valid_metric_rows) >= 2:
    pearson = valid_metric_rows["dfsi_calculated"].corr(valid_metric_rows["dfsi_published"], method="pearson")
    spearman = valid_metric_rows["dfsi_calculated"].corr(valid_metric_rows["dfsi_published"], method="spearman")
else:
    pearson = np.nan
    spearman = np.nan

metrics = pd.DataFrame([
    {"metric": "comparison_rows", "value": len(comparison_table)},
    {"metric": "finite_metric_rows", "value": len(valid_metric_rows)},
    {"metric": "missing_component_rows", "value": int(comparison_table[component_columns].isna().any(axis=1).sum())},
    {"metric": "flood_event_unmatched_rows_filled_as_zero", "value": int(comparison_table["flood_events_missing_from_inventory"].sum())},
    {"metric": "max_logspace_vs_product_abs_diff", "value": comparison_table["logspace_product_abs_diff"].max()},
    {"metric": "pearson_correlation", "value": pearson},
    {"metric": "spearman_rank_correlation", "value": spearman},
    {"metric": "mean_absolute_difference", "value": valid_metric_rows["absolute_difference"].mean() if len(valid_metric_rows) else np.nan},
    {"metric": "median_absolute_difference", "value": valid_metric_rows["absolute_difference"].median() if len(valid_metric_rows) else np.nan},
    {"metric": "maximum_absolute_difference", "value": valid_metric_rows["absolute_difference"].max() if len(valid_metric_rows) else np.nan},
])

display(metrics)
metrics.to_csv(TABLE_DIR / "dfsi_reproduction_metrics.csv", index=False)

,metric,value
0,comparison_rows,7.200000e+02
1,finite_metric_rows,7.090000e+02
2,missing_component_rows,1.100000e+01
3,flood_event_unmatched_rows_filled_as_zero,8.600000e+01
4,max_logspace_vs_product_abs_diff,3.552714e-15
5,pearson_correlation,-9.344958e-01
6,spearman_rank_correlation,-9.344742e-01
7,mean_absolute_difference,2.363652e+01
8,median_absolute_difference,2.440347e+01
9,maximum_absolute_difference,3.435195e+01


## Largest Discrepancies

The table below lists up to 30 districts with the largest absolute difference among rows where the calculated DFSI is finite. If no finite values exist under the literal formula, the table will be empty and the blocker table will explain why.

In [13]:
largest_discrepancies = (
    comparison_table.dropna(subset=["absolute_difference"])
    .sort_values("absolute_difference", ascending=False)
    .head(30)
)

display(largest_discrepancies)
largest_discrepancies.to_csv(TABLE_DIR / "dfsi_largest_discrepancies.csv", index=False)

,district_review_key,Dist_Name_impact,DFSI_Dist_Name,State_Name,component_flood_events,component_mean_flood_duration,component_fatalities,component_injured,component_flooded_area,Percent_Flooded_Area,Parmanent_Water,component_population,dfsi_float64_product,dfsi_calculated,dfsi_calculated_from_product,logspace_product_abs_diff,dfsi_published,absolute_difference,rank_difference,flood_events_missing_from_inventory,lgd_codes_seen,lgd_code_count
671,north twenty four pargan*,North Twenty Four Pargan*,North Twenty Four Pargan*,WEST BENGAL,0.0,19.0,294.0,16.0,11.225591,18.664847,7.439256,10544399.0,2.621714e-16,-15.581415,-15.581415,0.000000e+00,18.770538,34.351952,704.0,True,NaN,NaN
104,patna,Patna,Patna,BIHAR,71.0,16.0,381.0,30.0,18.820619,20.269812,1.449193,6868950.0,6.876818e-15,-14.162612,-14.162612,1.776357e-15,19.300564,33.463177,701.0,False,520,1.0
78,purba champaran,Purba Champaran,Purba Champaran,BIHAR,0.0,19.0,184.0,20.0,9.387441,9.508604,0.121163,6322257.0,1.059318e-15,-14.974974,-14.974974,1.776357e-15,18.295788,33.270761,697.0,True,NaN,NaN
661,darjiling,Darjiling,Darjiling,WEST BENGAL,0.0,11.0,548.0,63.0,1.918361,2.014600,0.096239,1849559.0,8.003623e-16,-15.096713,-15.096713,0.000000e+00,18.018981,33.115695,686.0,True,NaN,NaN
665,murshidabad,Murshidabad,Murshidabad,WEST BENGAL,44.0,23.0,435.0,9.0,10.404534,11.638689,1.234154,8350690.0,1.035687e-14,-13.984772,-13.984772,0.000000e+00,18.910884,32.895655,699.0,False,319,1.0
159,ahmadabad,Ahmadabad,Ahmadabad,GUJARAT,0.0,12.0,177.0,23.0,4.099835,4.222293,0.122458,8738876.0,2.269832e-15,-14.644006,-14.644006,0.000000e+00,18.117122,32.761129,688.0,True,NaN,NaN
10,guntur,Guntur,Guntur,ANDHRA PRADESH,33.0,8.0,188.0,108.0,9.242779,10.242307,0.999528,5374652.0,1.762455e-14,-13.753882,-13.753882,0.000000e+00,18.832069,32.585951,694.0,False,506,1.0
673,medinipur west,Medinipur West,Medinipur West,WEST BENGAL,0.0,10.0,294.0,10.0,7.812548,7.838526,0.025978,5510554.0,1.955572e-15,-14.708726,-14.708726,1.776357e-15,17.869046,32.577772,679.0,True,NaN,NaN
389,thane,Thane,Thane,MAHARASHTRA,97.0,3.0,418.0,204.0,1.253628,3.515191,2.261563,9950177.0,3.035786e-14,-13.517729,-13.517729,0.000000e+00,18.861789,32.379518,692.0,False,497,1.0
664,maldah,Maldah,Maldah,WEST BENGAL,17.0,11.0,221.0,53.0,9.357036,11.776244,2.419208,4260890.0,1.277190e-14,-13.893744,-13.893744,0.000000e+00,18.258130,32.151874,686.0,False,,0.0


## Reproduction Blockers

These records summarise the issues that prevent exact reproduction. They are intentionally explicit because the source files do not share a complete, unambiguous district identifier.

In [14]:
blockers = pd.DataFrame([
    {
        "blocker_type": "missing_lgd_codes_in_summary_sources",
        "affected_records": len(impact) + len(area) + len(published),
        "details": "District_FloodImpact.csv, District_FloodedArea.csv, and DFSI.csv do not contain LGD district/state codes, preventing a fully identifier-based join.",
    },
    {
        "blocker_type": "repeated_district_names",
        "affected_records": len(multiply_matched_records),
        "details": "Some district names repeat across source files and cannot be safely joined by district name alone without state or LGD identifiers.",
    },
    {
        "blocker_type": "unmatched_join_keys",
        "affected_records": len(unmatched_records),
        "details": "Some district keys do not appear in every source needed for the reproduction or do not match any parsed flood event.",
    },
    {
        "blocker_type": "missing_numeric_components",
        "affected_records": int(comparison_table[component_columns].isna().any(axis=1).sum()),
        "details": "Rows with missing numeric components are preserved as missing and are not imputed.",
    },
    {
        "blocker_type": "event_inventory_parse_issues",
        "affected_records": len(event_parse_audit),
        "details": "Some event rows have missing district lists or district/code count mismatches after parsing comma-separated fields.",
    },
    {
        "blocker_type": "state_not_assignable_to_exploded_event_districts",
        "affected_records": len(flood_inventory[flood_inventory["State"].notna()]),
        "details": "The event inventory state field may list multiple states per event and is not positionally aligned to each exploded district entry.",
    },
])

display(blockers)
blockers.to_csv(TABLE_DIR / "dfsi_reproduction_blockers.csv", index=False)

,blocker_type,affected_records,details
0,missing_lgd_codes_in_summary_sources,2208,"District_FloodImpact.csv, District_FloodedArea.csv, and DFSI.csv do not contain LGD district/state codes, preventing a fully identifier-based join."
1,repeated_district_names,6,Some district names repeat across source files and cannot be safely joined by district name alone without state or LGD identifiers.
2,unmatched_join_keys,323,Some district keys do not appear in every source needed for the reproduction or do not match any parsed flood event.
3,missing_numeric_components,11,Rows with missing numeric components are preserved as missing and are not imputed.
4,event_inventory_parse_issues,59,Some event rows have missing district lists or district/code count mismatches after parsing comma-separated fields.
5,state_not_assignable_to_exploded_event_districts,6876,The event inventory state field may list multiple states per event and is not positionally aligned to each exploded district entry.


## Saved Outputs

The reproduction attempt writes all audit and comparison tables to `outputs/tables`.

In [15]:
output_files = [
    "dfsi_component_mapping.csv",
    "dfsi_join_documentation.csv",
    "dfsi_event_parse_audit.csv",
    "dfsi_inventory_name_lgd_ambiguities.csv",
    "dfsi_unmatched_records.csv",
    "dfsi_multiply_matched_records.csv",
    "dfsi_reproduction_comparison.csv",
    "dfsi_reproduction_metrics.csv",
    "dfsi_largest_discrepancies.csv",
    "dfsi_reproduction_blockers.csv",
]

print("Saved DFSI reproduction outputs:")
for file_name in output_files:
    path = TABLE_DIR / file_name
    print(f"- {path.relative_to(PROJECT_ROOT)} ({path.stat().st_size:,} bytes)")

Saved DFSI reproduction outputs:
- outputs/tables/dfsi_component_mapping.csv (1,722 bytes)
- outputs/tables/dfsi_join_documentation.csv (1,268 bytes)
- outputs/tables/dfsi_event_parse_audit.csv (3,890 bytes)
- outputs/tables/dfsi_inventory_name_lgd_ambiguities.csv (88 bytes)
- outputs/tables/dfsi_unmatched_records.csv (33,972 bytes)
- outputs/tables/dfsi_multiply_matched_records.csv (847 bytes)
- outputs/tables/dfsi_reproduction_comparison.csv (165,000 bytes)
- outputs/tables/dfsi_reproduction_metrics.csv (410 bytes)
- outputs/tables/dfsi_largest_discrepancies.csv (7,503 bytes)
- outputs/tables/dfsi_reproduction_blockers.csv (976 bytes)
